# Shared Abstract Computation — Part 1: controlled from-scratch study

Small transformers are trained from scratch on synthetic ordinal tasks while varying how many domains they see. This tests the **cause** behind the thesis: whether a shared mid-depth band forms, when it forms, and whether it predicts generalisation to untrained operation–domain cells.

| Hypothesis | What this notebook computes |
|---|---|
| **Ground-truth gate** (hard gate) | Planted-shared vs planted-separate models. The Abstraction Index must separate them and track the planted share. The main grid refuses to start until this passes. |
| **H1** sharing grows with K | Abstraction Index (A) and peak cross-domain transfer (B) against K, p and size, over seeds |
| **H2** the band forms abruptly, with generalisation | Band strength and held-out-cell accuracy at ~20 log-spaced checkpoints; change-point onsets compared |
| **H4 in miniature** | Parametric → in-context transfer (B) and subspace transfer into in-context lists (C), against p |
| **H5, Part 1 version** | Model-level transfer against held-out-cell accuracy across models. Spearman, then partial Spearman controlling for K, p, size, trained-cell accuracy and two geometry baselines |

**Running it on Colab free tier**
- Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Results go to Google Drive, one file per model. If the session dies, rerun everything: finished models are skipped, so the grid resumes where it stopped. `TIME_BUDGET_MIN` stops new models before Colab's limit.
- `GRID = "pilot"` (default) runs 16 models: K ∈ {2,4,8,16} × p ∈ {0, 0.5} × small × 2 seeds. `GRID = "full"` runs the blueprint's 120. That is several sessions on the free tier, and resuming handles it.
- **Runtimes are untested on a GPU.** The pipeline was tested end to end on CPU. My T4 estimates are about 10–15 minutes for the gate and 2–4 minutes per small model, so the pilot takes roughly 1–1.5 hours. Large models take maybe 3–4× longer.

**Departures from the blueprint, decided while building and testing this**
1. **The planted "shared module" is shared *per operation*.** A middle block shared by every cell raises the generic baseline G as much as it raises S_op, so it plants no operation-organised sharing. Here, a fraction f of middle-layer heads and neurons is routed by operation only (shared across domains). The rest is routed by (operation, domain), and the first and last layers are routed by domain. f = 1 is planted-shared, f = 0 is planted-separate, and intermediate f values test whether the index tracks the share.
2. **DAS runs at two positions: the operand token and the final position.** In testing, small models copied the operand into later positions within block 0. After that, interventions at the operand do nothing even at full rank.
3. **Every DAS target gets a within-domain full-rank ceiling.** This is a complete swap from a source item in the *target* domain, and it bounds what any subspace at that location can do. A cross-domain full swap also transfers the source domain's identity, so it can't serve as a ceiling. The Experiment 1 pilot notebook made that mistake and needs the same fix.
4. **The default DAS rank is 4.** Rank 1 could not carry 12–20-way rank identity even within the anchor domain in testing.
5. **There is one prompt format, so no template control.** Replicate circuits come from disjoint sets of clean prompts.
6. **The 90% gate on trained cells uses training prompts.** Succession and predecessor cells have only 11–19 prompts, so held-out *cells* are the generalisation test.

## Configuration

In [ ]:
GRID = "pilot"                 # "pilot" (16 models) or "full" (120 models)
TIME_BUDGET_MIN = 300          # stop starting new models after this many minutes; rerun to resume
USE_DRIVE = True               # store results on Google Drive so runs survive disconnects
RESULTS_SUBDIR = "shared_abstraction_part1"
OVERRIDE_GATE = False          # True runs the grid even if the ground-truth gate fails (results then unvalidated)

# --- the grid (blueprint Part 1)
if GRID == "full":
    KS, PS, SIZES_RUN, SEEDS = [2, 4, 8, 16], [0.0, 0.25, 0.5], ["small", "large"], [0, 1, 2, 3, 4]
else:
    KS, PS, SIZES_RUN, SEEDS = [2, 4, 8, 16], [0.0, 0.5], ["small"], [0, 1]
SIZES = {"small": dict(L=4, d=256, H=8, lr=1e-3), "large": dict(L=8, d=512, H=8, lr=5e-4)}
TRAIN_STEPS = {"small": 12000, "large": 12000}
BATCH = 256
N_CKPT = 20                    # log-spaced checkpoints for H2 trajectories
CTX_N = 6                      # items per in-context list
ACC_GATE = 0.90                # a cell enters A/B/C only at >= 90% accuracy

# --- measurements
EAP_REPS, EAP_PER_REP, EAP_CORR, EAP_IG_STEPS = 5, 4, 4, 5
B_N_BASE, B_MAX_TARGETS, B_MAX_SOURCES = 24, 6, 3
TRAJ_B = dict(n_base=12, max_targets=3, max_sources=2)   # cheaper B at every checkpoint
DAS_RANK, DAS_STEPS, DAS_LR, DAS_TEST_N = 4, 200, 1e-2, 48
MIN_WITHIN, T_THRESH = 0.20, 0.50

# --- ground-truth gate
GATE_K, GATE_SIZE = 2, dict(L=4, d=192, H=24, lr=1e-3)   # 24 heads split cleanly into 3 ops x 2 domains at every f
GATE_FS, GATE_SEEDS, GATE_STEPS = [0.0, 0.25, 0.5, 0.75, 1.0], [0, 1, 2], 4000
GATE_SEPARATION = "every f=1 model above every f=0 model"
GATE_MIN_SPEARMAN = 0.80

In [ ]:
import os, json, time, math, random
import numpy as np
import torch
import matplotlib.pyplot as plt
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

RESULTS_DIR = RESULTS_SUBDIR
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = f"/content/drive/MyDrive/{RESULTS_SUBDIR}"
    except Exception as e:
        print(f"Drive not available ({type(e).__name__}); saving locally to ./{RESULTS_SUBDIR}. Results won't survive a disconnect.")
os.makedirs(f"{RESULTS_DIR}/gate", exist_ok=True); os.makedirs(f"{RESULTS_DIR}/grid", exist_ok=True)
print("Results directory:", RESULTS_DIR)
T_SESSION = time.time()

def to_json(o):
    if isinstance(o, dict): return {str(k): to_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_json(v) for v in o]
    if isinstance(o, np.ndarray): return to_json(o.tolist())
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o

def save(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh: json.dump(to_json(obj), fh)
    os.replace(tmp, path)       # atomic, so a disconnect never leaves a half-written result

## Library: world, model, training, measurements A / B / C
- **World.** There are 16 disjoint domains, each a fixed random order of 12–20 tokens. Domain d is the same in every model with the same seed, so K = 2 and K = 16 share domains 0–1. A pool of 64 extra tokens supplies in-context lists.
- **Prompts.** Parametric prompts are `[BOS, OP, x, (y), EQ] → answer`. In-context prompts are `[BOS, CTX, s1…s6, OP, x, (y), EQ] → answer`. Comparison uses gaps of 1–5 and both answer positions, and there is no wrap-around.
- **Model.** A pre-LN GPT with per-head input copies, so every head is its own EAP receiver. Optional routing masks build the planted models.
- **A.** EAP-IG edge scores, with resample-ablation counterfactuals from the same cell and 5 replicate circuits per cell from disjoint clean prompts. Weighted Jaccard is corrected by the noise ceiling, giving S_op, S_dom, G and the Abstraction Index. A top-k sweep is also reported.
- **B.** Operation-swap patching (succ ↔ pred) at the final position for every residual site. T(ℓ) = cross / within, defined only where the within effect is ≥ 20%.
- **C.** DAS on domain 0 succession at the operand and at the final position, tested zero-shot on other domains, on in-context lists and on predecessor. Controls are a random subspace, shuffled labels and the within-domain ceiling.

In [ ]:
# ============================================================================
# Part 1 library: synthetic world, model, training, measurements A/B/C
# ============================================================================
import math, json, os, time, random
from itertools import combinations
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------------------------------------------- vocabulary
PAD, BOS, EQ, CTXT, SUCC, PRED, CMP = range(7)
OPS = ["succ", "pred", "cmp"]
OP_TOK = {"succ": SUCC, "pred": PRED, "cmp": CMP}
MAX_K, SLOT, N_SPECIAL, CTX_POOL = 16, 20, 7, 64
POOL_START = N_SPECIAL + MAX_K * SLOT
VOCAB = POOL_START + CTX_POOL
CMP_MAX_GAP = 5


class World:
    """MAX_K disjoint domains, each a fixed random order of 12-20 tokens, plus a pool for in-context orders.
    The world depends only on the seed, so domain d is identical in every model trained with that seed."""
    def __init__(self, seed, n_range=(12, 20), ctx_n=6):
        rng = np.random.default_rng(10_000 + seed)
        self.domains = []
        for d in range(MAX_K):
            n = int(rng.integers(n_range[0], n_range[1] + 1))
            self.domains.append([int(N_SPECIAL + d * SLOT + s) for s in rng.permutation(SLOT)[:n]])
        self.pool = list(range(POOL_START, POOL_START + CTX_POOL))
        self.ctx_n = ctx_n

    def items(self, dom, seq=None):
        return seq if dom == "ctx" else self.domains[dom]


def valid_ranks(op, n):
    if op == "succ": return list(range(n - 1))
    if op == "pred": return list(range(1, n))
    return list(range(n))


def apply_op(op, i, j=None):
    return i + 1 if op == "succ" else (i - 1 if op == "pred" else max(i, j))


def make_prompt(world, op, dom, i, j=None, seq=None):
    """Parametric: [BOS, OP, x, (y), EQ].  In-context: [BOS, CTXT, s1..sn, OP, x, (y), EQ].  Answer follows EQ."""
    items = world.items(dom, seq)
    ids = [BOS] if dom != "ctx" else [BOS, CTXT] + list(seq)
    ids += [OP_TOK[op], items[i]]
    opnd = len(ids) - 1
    if op == "cmp": ids.append(items[j])
    ids.append(EQ)
    r = apply_op(op, i, j)
    return dict(ids=ids, q=len(ids) - 1, ans=items[r], ans_rank=r, op=op, dom=dom, i=i, j=j,
                opnd=opnd, seq=list(seq) if seq is not None else None)


def sample_seq(world, rng):
    return [int(x) for x in rng.choice(world.pool, world.ctx_n, replace=False)]


def cell_prompts(world, op, dom, rng=None, n_ctx=200):
    if dom == "ctx":
        out = []
        for _ in range(n_ctx):
            seq = sample_seq(world, rng); n = len(seq)
            if op == "cmp":
                i, j = rng.choice(n, 2, replace=False)
            else:
                i, j = int(rng.choice(valid_ranks(op, n))), None
            out.append(make_prompt(world, op, "ctx", int(i), None if j is None else int(j), seq=seq))
        return out
    n = len(world.domains[dom])
    if op == "cmp":
        return [make_prompt(world, op, dom, i, j) for i in range(n) for j in range(n)
                if i != j and abs(i - j) <= CMP_MAX_GAP]
    return [make_prompt(world, op, dom, i) for i in valid_ranks(op, n)]


def tens(ps, K):
    """Prompts -> padded tensors (right-padding is harmless: the answer is read at q under causal attention)."""
    T = max(len(p["ids"]) for p in ps)
    ids = torch.full((len(ps), T), PAD, dtype=torch.long)
    for b, p in enumerate(ps): ids[b, :len(p["ids"])] = torch.tensor(p["ids"])
    f = lambda k: torch.tensor([p[k] for p in ps], dtype=torch.long)
    return dict(ids=ids.to(DEV), q=f("q").to(DEV), ans=f("ans").to(DEV), opnd=f("opnd").to(DEV),
                op=torch.tensor([OPS.index(p["op"]) for p in ps], device=DEV),
                dom=torch.tensor([K if p["dom"] == "ctx" else p["dom"] for p in ps], device=DEV))


# ---------------------------------------------------------------- training data
class Sampler:
    """Uniform over trained parametric cells with prob 1-p; in-context stream with prob p."""
    def __init__(self, world, K, train_cells, p_ctx, seed):
        self.world, self.K, self.p = world, K, p_ctx
        self.g = torch.Generator().manual_seed(seed)
        rows, starts, counts = [], [], []
        for (op, d) in train_cells:
            ps = cell_prompts(world, op, d)
            starts.append(len(rows)); counts.append(len(ps)); rows += ps
        self.P = tens(rows, K) if rows else None
        self.starts = torch.tensor(starts, device=DEV); self.counts = torch.tensor(counts, device=DEV)
        self.n_cells = len(train_cells)

    def ctx_batch(self, m):
        n = self.world.ctx_n
        seqs = POOL_START + torch.argsort(torch.rand(m, CTX_POOL, generator=self.g), 1)[:, :n]
        op = torch.randint(0, 3, (m,), generator=self.g)
        u = torch.rand(m, generator=self.g); u2 = torch.rand(m, generator=self.g)
        i = torch.where(op == 0, (u * (n - 1)).long(), torch.where(op == 1, 1 + (u * (n - 1)).long(), (u * n).long()))
        j = (i + 1 + (u2 * (n - 1)).long()) % n
        r = torch.where(op == 0, i + 1, torch.where(op == 1, i - 1, torch.maximum(i, j)))
        ar = torch.arange(m)
        T = n + 6
        ids = torch.full((m, T), PAD, dtype=torch.long)
        ids[:, 0], ids[:, 1] = BOS, CTXT
        ids[:, 2:2 + n] = seqs
        ids[:, 2 + n] = torch.tensor([SUCC, PRED, CMP])[op]
        ids[:, 3 + n] = seqs[ar, i]
        is_cmp = op == 2
        ids[is_cmp, 4 + n] = seqs[ar, j][is_cmp]
        q = torch.where(is_cmp, torch.full_like(op, 5 + n), torch.full_like(op, 4 + n))
        ids[ar, q] = EQ
        return dict(ids=ids.to(DEV), q=q.to(DEV), ans=seqs[ar, r].to(DEV), op=op.to(DEV),
                    dom=torch.full((m,), self.K, device=DEV))

    def batch(self, B):
        m_ctx = int(round(self.p * B)) if self.n_cells else B
        m_par = B - m_ctx
        parts = []
        if m_par:
            c = torch.randint(0, self.n_cells, (m_par,), generator=self.g).to(DEV)
            off = (torch.rand(m_par, generator=self.g).to(DEV) * self.counts[c]).long()
            idx = self.starts[c] + off
            parts.append({k: self.P[k][idx] for k in ("ids", "q", "ans", "op", "dom")})
        if m_ctx: parts.append(self.ctx_batch(m_ctx))
        T = max(x["ids"].shape[1] for x in parts)
        for x in parts:
            if x["ids"].shape[1] < T:
                x["ids"] = F.pad(x["ids"], (0, T - x["ids"].shape[1]), value=PAD)
        return {k: torch.cat([x[k] for x in parts]) for k in ("ids", "q", "ans", "op", "dom")}


# ---------------------------------------------------------------- model
class Block(nn.Module):
    def __init__(self, d, H, dm, L):
        super().__init__()
        self.H, self.dh = H, d // H
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s = 0.02
        self.WQ = nn.Parameter(torch.randn(H, d, self.dh) * s)
        self.WK = nn.Parameter(torch.randn(H, d, self.dh) * s)
        self.WV = nn.Parameter(torch.randn(H, d, self.dh) * s)
        self.WO = nn.Parameter(torch.randn(H, self.dh, d) * s / math.sqrt(2 * L))
        self.fc1, self.fc2 = nn.Linear(d, dm), nn.Linear(dm, d)
        nn.init.normal_(self.fc1.weight, std=s); nn.init.normal_(self.fc2.weight, std=s / math.sqrt(2 * L))
        nn.init.zeros_(self.fc1.bias); nn.init.zeros_(self.fc2.bias)

    def forward(self, x, hmask, mmask, rec):
        B, T, d = x.shape
        grad = rec is not None and rec.get("grad")
        xh = x.unsqueeze(2).expand(B, T, self.H, d)
        if grad:   # one input copy per head so each head is its own EAP receiver
            gh = torch.zeros(B, T, self.H, d, device=x.device, requires_grad=True); rec["recv"].append(gh); xh = xh + gh
        h = self.ln1(xh)
        q = torch.einsum("bthd,hde->bhte", h, self.WQ)
        k = torch.einsum("bthd,hde->bhte", h, self.WK)
        v = torch.einsum("bthd,hde->bhte", h, self.WV)
        att = (q @ k.transpose(-1, -2)) / math.sqrt(self.dh)
        causal = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        att = att.masked_fill(~causal, float("-inf")).softmax(-1)
        oh = torch.einsum("bhte,hed->bthd", att @ v, self.WO)
        if hmask is not None: oh = oh * hmask[:, None, :, None]
        if rec is not None and "send" in rec: rec["send"].append(oh)
        x = x + oh.sum(2)
        xm = x
        if grad:
            gm = torch.zeros_like(x, requires_grad=True); rec["recv"].append(gm); xm = xm + gm
        a = F.gelu(self.fc1(self.ln2(xm)))
        if mmask is not None: a = a * mmask[:, None, :]
        mo = self.fc2(a)
        if rec is not None and "send" in rec: rec["send"].append(mo)
        return x + mo


class Model(nn.Module):
    def __init__(self, L, d, H, dm=None, n_ctx=16):
        super().__init__()
        self.L, self.d, self.H, self.dm = L, d, H, dm or 4 * d
        self.tok, self.pos = nn.Embedding(VOCAB, d), nn.Embedding(n_ctx, d)
        nn.init.normal_(self.tok.weight, std=0.02); nn.init.normal_(self.pos.weight, std=0.02)
        self.blocks = nn.ModuleList([Block(d, H, self.dm, L) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        self.unembed = nn.Linear(d, VOCAB, bias=False)
        self.route = None          # planted models: (head_table [3, K, L, H], mlp_table [3, K, L, dm])

    def embed(self, ids):
        return self.tok(ids) + self.pos(torch.arange(ids.shape[1], device=ids.device))

    def masks(self, op, dom):
        if self.route is None: return None, None
        ht, mt = self.route
        return ht[op, dom], mt[op, dom]

    def forward(self, ids, q, op=None, dom=None, embed=None, patch=None, cache=False, rec=None):
        hm, mm = self.masks(op, dom) if self.route is not None else (None, None)
        x = self.embed(ids) if embed is None else embed
        if rec is not None and "send" in rec: rec["send"].append(x)
        sites = []
        for l, blk in enumerate(self.blocks):
            if patch and l in patch: x = patch[l](x)
            if cache: sites.append(x)
            x = blk(x, None if hm is None else hm[:, l], None if mm is None else mm[:, l], rec)
        if patch and self.L in patch: x = patch[self.L](x)
        if cache: sites.append(x)
        if rec is not None and rec.get("grad"):
            gf = torch.zeros_like(x, requires_grad=True); rec["recv"].append(gf); x = x + gf
        ar = torch.arange(ids.shape[0], device=ids.device)
        out = self.unembed(self.lnf(x[ar, q]))
        return (out, sites) if cache else out


def run(model, T, **kw):
    return model(T["ids"], T["q"], T["op"], T["dom"], **kw)


@torch.no_grad()
def accuracy(model, ps, K, bs=512):
    if not ps: return float("nan")
    hits = []
    for s in range(0, len(ps), bs):
        T = tens(ps[s:s + bs], K)
        hits.append((run(model, T).argmax(-1) == T["ans"]).float())
    return torch.cat(hits).mean().item()


# ---------------------------------------------------------------- planted routing (ground-truth gate)
def planted_route(L, H, dm, K, f, n_groups_check=True):
    """Layer 0 and layer L-1: routed by domain (shared across operations).
    Middle layers: a fraction f of heads/neurons routed by operation only (shared across domains, the planted
    abstract computation); the rest routed by (operation, domain) (separate per-domain pathways)."""
    ht = torch.zeros(3, K, L, H); mt = torch.zeros(3, K, L, dm)
    def split(n, k):     # k equal contiguous groups over n units
        assert n % k == 0, f"{n} units cannot be split into {k} groups"
        return [list(range(g * n // k, (g + 1) * n // k)) for g in range(k)]
    for l in range(L):
        if l in (0, L - 1):
            for d, (hg, mg) in enumerate(zip(split(H, K), split(dm, K))):
                ht[:, d, l, hg] = 1; mt[:, d, l, mg] = 1
            continue
        nh_sh, nm_sh = int(round(f * H)), int(round(f * dm))
        if nh_sh:
            for o, (hg, mg) in enumerate(zip(split(nh_sh, 3), split(nm_sh, 3))):
                ht[o, :, l, hg] = 1; mt[o, :, l, mg] = 1
        if H - nh_sh:
            hs = split(H - nh_sh, 3 * K); ms = split(dm - nm_sh, 3 * K)
            for o in range(3):
                for d in range(K):
                    g = o * K + d
                    ht[o, d, l, [nh_sh + x for x in hs[g]]] = 1
                    mt[o, d, l, [nm_sh + x for x in ms[g]]] = 1
    return ht.to(DEV), mt.to(DEV)


# ---------------------------------------------------------------- training
def log_checkpoints(total, n, first=50):
    pts = np.unique(np.round(np.geomspace(first, total, n)).astype(int))
    return [int(x) for x in pts]


def train(model, sampler, steps, lr, batch, checkpoints=(), on_ckpt=None, warmup=200, log_every=0):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) *
                                              (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, s / steps)))))
    use_amp = DEV == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ck = set(checkpoints); t0 = time.time()
    for step in range(1, steps + 1):
        b = sampler.batch(batch)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(run(model, b), b["ans"])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        if log_every and step % log_every == 0:
            print(f"    step {step:6d} loss {loss.item():.3f} [{time.time() - t0:.0f}s]", flush=True)
        if step in ck and on_ckpt is not None:
            model.eval(); on_ckpt(step, model); model.train()
    model.eval()
    return model


# ---------------------------------------------------------------- measurement A: EAP-IG + Abstraction Index
def node_names(L, H):
    snd = [("emb", -1)]; rcv = []
    for l in range(L):
        snd += [(f"a{l}.{h}", 2 * l) for h in range(H)] + [(f"m{l}", 2 * l + 1)]
        rcv += [(f"a{l}.{h}", 2 * l) for h in range(H)] + [(f"m{l}", 2 * l + 1)]
    rcv.append(("out", 2 * L))
    valid = torch.tensor([[s[1] < r[1] for r in rcv] for s in snd])
    return snd, rcv, valid


def _flat(parts):
    out = []
    for t in parts:
        out.append(t.permute(2, 0, 1, 3) if t.dim() == 4 else t.unsqueeze(0))
    return torch.cat(out)


def eap_ig(model, C, X, ans_x, steps=5):
    """Edge scores (sender output change clean-vs-corrupt) . (gradient of logit diff at receiver input,
    integrated along the input-embedding path from corrupt to clean). Returns a vector over valid edges."""
    _, _, valid = node_names(model.L, model.H)
    with torch.no_grad():
        rc, rx = {"send": []}, {"send": []}
        run(model, C, rec=rc); run(model, X, rec=rx)
        D = _flat(rc["send"]) - _flat(rx["send"])
        e_c, e_x = model.embed(C["ids"]), model.embed(X["ids"])
    G = None
    ar = torch.arange(C["ids"].shape[0], device=DEV)
    for k in range(steps):
        a = (k + 0.5) / steps
        rec = {"grad": True, "recv": []}
        with torch.enable_grad():
            lg = run(model, C, embed=(e_x + a * (e_c - e_x)).detach(), rec=rec)
            m = (lg[ar, C["ans"]] - lg[ar, ans_x]).sum()
            g = torch.autograd.grad(m, rec["recv"])
        g = _flat(g)
        G = g if G is None else G + g
    S = torch.einsum("sbtd,rbtd->sr", D.float(), (G / steps).float()) / C["ids"].shape[0]
    return S[valid.to(DEV)].detach().cpu().numpy()


def cell_pairs_for_eap(world, op, dom, rng, n_rep, per_rep, n_corr):
    """Replicates are disjoint sets of clean prompts; each clean prompt is paired with corrupted prompts from the
    same cell that have a different answer (resample ablation)."""
    reps = []
    if dom == "ctx":
        for _ in range(n_rep):
            cl, cx = [], []
            for _ in range(per_rep * n_corr):
                seq = sample_seq(world, rng)
                while True:
                    a = cell_from_seq(world, op, seq, rng); b = cell_from_seq(world, op, seq, rng)
                    if a["ans"] != b["ans"]: break
                cl.append(a); cx.append(b)
            reps.append((cl, cx))
        return reps
    ps = cell_prompts(world, op, dom)
    order = rng.permutation(len(ps))
    groups = np.array_split(order, n_rep)
    for g in groups:
        cl, cx = [], []
        for idx in g:
            c = ps[idx]
            others = [p for p in ps if p["ans"] != c["ans"]]
            for oi in rng.choice(len(others), min(n_corr, len(others)), replace=False):
                cl.append(c); cx.append(others[oi])
        reps.append((cl, cx))
    return reps


def cell_from_seq(world, op, seq, rng):
    n = len(seq)
    if op == "cmp":
        i, j = rng.choice(n, 2, replace=False); return make_prompt(world, op, "ctx", int(i), int(j), seq=seq)
    return make_prompt(world, op, "ctx", int(rng.choice(valid_ranks(op, n))), seq=seq)


def measure_A(model, world, K, cells, rng, n_rep=5, per_rep=6, n_corr=4, ig_steps=5):
    """Returns {cell: [replicate edge-score vectors]}."""
    model.requires_grad_(False)
    out = {}
    for (op, dom) in cells:
        vecs = []
        for cl, cx in cell_pairs_for_eap(world, op, dom, rng, n_rep, per_rep, n_corr):
            if not cl: continue
            C, X = tens(cl, K), tens(cx, K)
            vecs.append(np.abs(eap_ig(model, C, X, X["ans"], ig_steps)))
        out[(op, dom)] = vecs
    return out


def wjacc(a, b):
    den = np.maximum(a, b).sum()
    return float(np.minimum(a, b).sum() / den) if den > 0 else np.nan


def topk_jacc(a, b, k):
    k = min(k, len(a))
    A = set(np.argsort(-a)[:k]); Bs = set(np.argsort(-b)[:k])
    return len(A & Bs) / len(A | Bs)


def abstraction_index(circ, kind="soft", k=None):
    """Noise-corrected overlap classes and AI.  circ: {(op, dom): [replicate vectors]}."""
    sim = (lambda a, b: wjacc(a, b)) if kind == "soft" else (lambda a, b: topk_jacc(a, b, k))
    cells = [c for c in circ if len(circ[c]) >= 2]
    R = {c: np.nanmean([sim(a, b) for a, b in combinations(circ[c], 2)]) for c in cells}
    cls = {"S_op": [], "S_dom": [], "G": []}
    for c1, c2 in combinations(cells, 2):
        o = np.nanmean([sim(a, b) for a in circ[c1] for b in circ[c2]])
        o_t = o / math.sqrt(R[c1] * R[c2]) if R[c1] > 0 and R[c2] > 0 else np.nan
        same_op, same_dom = c1[0] == c2[0], c1[1] == c2[1]
        if same_op and not same_dom: cls["S_op"].append(o_t)
        elif same_dom and not same_op: cls["S_dom"].append(o_t)
        elif not same_op and not same_dom: cls["G"].append(o_t)
    m = {k_: float(np.nanmean(v)) if v else np.nan for k_, v in cls.items()}
    d_op, d_dom = m["S_op"] - m["G"], m["S_dom"] - m["G"]
    # AI leaves [0, 1] when one of the deltas is negative (e.g. same-domain pairs overlap *less* than the generic
    # baseline). It is clipped to its interpretable range; AI_raw keeps the unclipped value.
    ai_raw = d_op / (d_op + d_dom) if (d_op + d_dom) > 1e-9 else np.nan
    ai = float(np.clip(ai_raw, 0, 1)) if np.isfinite(ai_raw) else np.nan
    return dict(AI=ai, AI_raw=ai_raw, clipped=bool(np.isfinite(ai_raw) and not 0 <= ai_raw <= 1), d_op=d_op, d_dom=d_dom, **m, R_mean=float(np.nanmean(list(R.values()))) if R else np.nan,
                R_min=float(np.nanmin(list(R.values()))) if R else np.nan)


# ---------------------------------------------------------------- measurement B: operation-swap patching
SWAPS = [("succ", "pred"), ("pred", "succ")]


def _patch_at(pos, vecs):
    def f(x):
        x = x.clone(); x[torch.arange(x.shape[0], device=x.device), pos] = vecs.to(x.dtype); return x
    return f


@torch.no_grad()
def opswap_group(model, world, K, B, X, Y, sources, n_base, rng):
    """Base: domain B, op X, item b. Within source: domain B, op Y, same b (same list for in-context).
    Cross source: domain A, op Y, rank-matched item when valid. Patch the final-position residual at each site."""
    nB = world.ctx_n if B == "ctx" else len(world.domains[B])
    bases, wins, cross = [], [], {A: [] for A in sources}
    for _ in range(n_base):
        i = int(rng.integers(1, nB - 1)); seq = sample_seq(world, rng) if B == "ctx" else None
        bases.append(make_prompt(world, X, B, i, seq=seq)); wins.append(make_prompt(world, Y, B, i, seq=seq))
        for A in sources:
            nA = world.ctx_n if A == "ctx" else len(world.domains[A])
            ok = [r for r in range(nA) if 0 <= apply_op(Y, r) < nA]
            r = i if i in ok else int(rng.choice(ok))
            cross[A].append(make_prompt(world, Y, A, r, seq=sample_seq(world, rng) if A == "ctx" else None))
    Tb, Tw = tens(bases, K), tens(wins, K)
    ar = torch.arange(n_base, device=DEV)
    yb = torch.tensor([w["ans"] for w in wins], device=DEV); xb = Tb["ans"]
    ld = lambda lg: (lg[ar, yb] - lg[ar, xb]).float().cpu().numpy()
    LD0 = ld(run(model, Tb))
    lw, sw = run(model, Tw, cache=True)
    full = ld(lw) - LD0
    Tc = {A: tens(cross[A], K) for A in sources}
    sc = {A: run(model, Tc[A], cache=True)[1] for A in sources}
    nS = model.L + 1
    dW = np.zeros((nS, n_base)); dC = {A: np.zeros((nS, n_base)) for A in sources}
    for l in range(nS):
        dW[l] = ld(run(model, Tb, patch={l: _patch_at(Tb["q"], sw[l][ar, Tw["q"]])})) - LD0
        for A in sources:
            dC[A][l] = ld(run(model, Tb, patch={l: _patch_at(Tb["q"], sc[A][l][ar, Tc[A]["q"]])})) - LD0
    return [dict(B=str(B), X=X, A=str(A), dC=dC[A][:, n], dW=dW[:, n], full=float(full[n]))
            for A in sources for n in range(n_base)]


def profile(rows, min_within=0.2, min_full=0.5):
    """T(l) = mean cross / mean within effect, defined only where the within patch moves >= min_within of the full
    effect, and only if the full effect itself is >= min_full logits (otherwise the ratios are noise)."""
    if not rows: return None
    C = np.stack([r["dC"] for r in rows]); W = np.stack([r["dW"] for r in rows]); Fu = np.array([r["full"] for r in rows])
    if Fu.mean() < min_full:
        nanv = np.full(W.shape[1], np.nan)
        return dict(eW=nanv, eC=nanv, T=nanv, full=float(Fu.mean()))
    eW = W.mean(0) / Fu.mean(); eC = C.mean(0) / Fu.mean()
    T = np.where(eW >= min_within, C.mean(0) / np.where(np.abs(W.mean(0)) > 1e-9, W.mean(0), np.nan), np.nan)
    return dict(eW=eW, eC=eC, T=T, full=float(Fu.mean()))


def peak(T):
    return float(np.nanmax(T)) if T is not None and np.any(~np.isnan(T)) else 0.0


def band(T, thr=0.5):
    ok = np.where(np.nan_to_num(T, nan=-1) >= thr)[0]
    return (int(ok[0]), int(ok[-1])) if len(ok) else (None, None)


def measure_B(model, world, K, ok_cells, rng, n_base=24, max_targets=6, max_sources=3, include_ctx=True):
    """Returns rows for param->param and param->ctx transfer. Targets need both operations; a source only needs
    the operation being swapped in."""
    par = [d for d in range(K) if ("succ", d) in ok_cells and ("pred", d) in ok_cells]
    any_par = list(range(K))
    rows_pp, rows_pc = [], []
    targets = list(rng.permutation(par)[:max_targets]) if par else []
    for B in targets:
        for X, Y in SWAPS:
            srcs = [A for A in any_par if A != B and (Y, A) in ok_cells]
            srcs = [int(a) for a in rng.permutation(srcs)[:max_sources]]
            if srcs: rows_pp += opswap_group(model, world, K, int(B), X, Y, srcs, n_base, rng)
    if include_ctx and ("succ", "ctx") in ok_cells and ("pred", "ctx") in ok_cells:
        for X, Y in SWAPS:
            srcs = [int(a) for a in rng.permutation([A for A in any_par if (Y, A) in ok_cells])[:max_sources]]
            if srcs: rows_pc += opswap_group(model, world, K, "ctx", X, Y, srcs, n_base, rng)
    return rows_pp, rows_pc


# ---------------------------------------------------------------- measurement C: DAS (shared ordinal variable)
# The variable "ordinal rank of the operand" is localised at two positions: the operand token itself and the
# final (EQ) position, which the operand is copied into within the first block or two. Every (position, site)
# gets a within-target-domain full-rank ceiling: a complete swap from a source item of the *target* domain.
# A cross-domain full swap would also carry the source domain's identity, so it is not a ceiling.
POSITIONS = ("opnd", "q")


def _das_patch(pos, src, Q):
    def f(x):
        x = x.clone(); ar = torch.arange(x.shape[0], device=x.device)
        h = x[ar, pos].float()
        x[ar, pos] = (h + (src - h) @ Q @ Q.T).to(x.dtype); return x
    return f


def das_sets(world, K, ok_cells, rng, test_n=48, max_test_domains=6):
    """Anchor = domain 0. Train/held-out pairs on anchor succession; zero-shot sets for other domains, ctx, pred.
    Each pair carries an anchor source prompt and a within-target-domain source prompt of the same rank."""
    n0 = len(world.domains[0])
    def pair(op, D, rb, rs, seq):
        return dict(base=make_prompt(world, op, D, rb, seq=seq), op=op, rs=rs,
                    src=make_prompt(world, op, 0, rs), wsrc=make_prompt(world, op, D, rs, seq=seq),
                    tgt=world.items(D, seq)[apply_op(op, rs)])
    allp = [pair("succ", 0, rb, rs, None) for rb in valid_ranks("succ", n0) for rs in valid_ranks("succ", n0) if rs != rb]
    order = rng.permutation(len(allp)); n_tr = int(0.7 * len(allp))
    train = [allp[k] for k in order[:n_tr]]
    tests = {("succ", 0): [allp[k] for k in order[n_tr:]]}
    doms = [int(d) for d in rng.permutation(list(range(1, K)))[:max_test_domains]] + ["ctx"]
    for op in ("succ", "pred"):
        if (op, 0) not in ok_cells: continue
        for D in ([0] if op == "pred" else []) + doms:
            if (op, D) not in ok_cells: continue
            nD = world.ctx_n if D == "ctx" else len(world.domains[D])
            out, tries = [], 0
            while len(out) < test_n and tries < test_n * 30:
                tries += 1
                rb = int(rng.choice(valid_ranks(op, nD))); rs = int(rng.choice(valid_ranks(op, n0)))
                if rs == rb or rs not in valid_ranks(op, nD): continue
                out.append(pair(op, D, rb, rs, sample_seq(world, rng) if D == "ctx" else None))
            tests[(op, D)] = out
    return train, tests


@torch.no_grad()
def attach_sources(model, pairs, K, bs=256):
    """Cache residuals of each pair's anchor source and within-domain source at both positions, all sites."""
    for s in range(0, len(pairs), bs):
        chunk = pairs[s:s + bs]
        for key in ("src", "wsrc"):
            T = tens([p[key] for p in chunk], K)
            _, sites = run(model, T, cache=True)
            ar = torch.arange(len(chunk), device=DEV)
            for pos in POSITIONS:
                V = torch.stack([st[ar, T[pos]] for st in sites], 1).float()   # [n, sites, d]
                for k, p in enumerate(chunk): p[f"{key}_{pos}"] = V[k]


def das_logits(model, pairs, site, pos, Q, K, key="src", grad=False):
    T = tens([p["base"] for p in pairs], K)
    src = torch.stack([p[f"{key}_{pos}"][site] for p in pairs])
    with torch.set_grad_enabled(grad):
        lg = run(model, T, patch={site: _das_patch(T[pos], src, Q)})
    return lg, torch.tensor([p["tgt"] for p in pairs], device=DEV)


@torch.no_grad()
def iia(model, pairs, site, pos, Q, K, key="src", bs=256):
    if not pairs: return np.array([])
    hits = []
    for s in range(0, len(pairs), bs):
        lg, tgt = das_logits(model, pairs[s:s + bs], site, pos, Q, K, key)
        hits.append((lg.argmax(-1) == tgt).float().cpu().numpy())
    return np.concatenate(hits)


def train_das(model, train_pairs, site, pos, K, rank, steps, lr, seed, shuffle=False, bs=64):
    g = torch.Generator().manual_seed(seed)
    W = torch.nn.Parameter((torch.randn(model.d, rank, generator=g) / math.sqrt(model.d)).to(DEV))
    opt = torch.optim.Adam([W], lr=lr)
    pairs = [dict(p) for p in train_pairs]
    if shuffle:
        tg = [p["tgt"] for p in pairs]; random.Random(seed).shuffle(tg)
        for p, t in zip(pairs, tg): p["tgt"] = t
    r = random.Random(seed)
    for _ in range(steps):
        b = r.sample(pairs, min(bs, len(pairs)))
        with torch.enable_grad():
            Q = torch.linalg.qr(W)[0]
            lg, tgt = das_logits(model, b, site, pos, Q, K, grad=True)
            loss = F.cross_entropy(lg.float(), tgt)
            opt.zero_grad(); loss.backward(); opt.step()
    return torch.linalg.qr(W.detach())[0]


def measure_C(model, world, K, ok_cells, rng, rank=1, steps=150, lr=1e-2, seed=0, test_n=48):
    """Per (position, site): DAS trained on anchor succession, IIA zero-shot elsewhere, random-subspace control,
    within-domain full-rank ceiling; shuffled-label control at the best (position, site)."""
    if ("succ", 0) not in ok_cells: return None
    model.requires_grad_(False)
    train_p, tests = das_sets(world, K, ok_cells, rng, test_n)
    attach_sources(model, train_p, K)
    for P in tests.values(): attach_sources(model, P, K)
    eye = torch.eye(model.d, device=DEV)
    gq = torch.Generator().manual_seed(999 + seed)
    res = {}
    for pos in POSITIONS:
        sites = range(model.L) if pos == "opnd" else range(1, model.L + 1)
        for site in sites:
            Q = train_das(model, train_p, site, pos, K, rank, steps, lr, seed)
            Qr = torch.linalg.qr(torch.randn(model.d, rank, generator=gq))[0].to(DEV)
            res[f"{pos}@{site}"] = {f"{op}/{D}": dict(
                das=float(iia(model, P, site, pos, Q, K).mean()), rand=float(iia(model, P, site, pos, Qr, K).mean()),
                ceil=float(iia(model, P, site, pos, eye, K, key="wsrc").mean()), n=len(P))
                for (op, D), P in tests.items() if P}
    best = max(res, key=lambda k: res[k]["succ/0"]["das"])
    pos, site = best.split("@"); site = int(site)
    Qs = train_das(model, train_p, site, pos, K, rank, steps, lr, seed, shuffle=True)
    for (op, D), P in tests.items():
        if P: res[best][f"{op}/{D}"]["shuf"] = float(iia(model, P, site, pos, Qs, K).mean())
    return dict(per_loc=res, best=best)


def rho_summary(C):
    """rho per zero-shot target at the best location: (IIA - rand) / (anchor held-out IIA - rand).
    A target is informative only if a full swap within its own domain works there (ceiling >= 0.5)."""
    if C is None: return {}
    r = C["per_loc"][C["best"]]
    den = r["succ/0"]["das"] - r["succ/0"]["rand"]
    out = {}
    for k, v in r.items():
        if k == "succ/0": continue
        rho = (v["das"] - v["rand"]) / den if den >= 0.1 else np.nan
        out[k] = dict(rho=rho, iia=v["das"], rand=v["rand"], ceil=v["ceil"], informative=bool(v["ceil"] >= 0.5))
    return out


# ---------------------------------------------------------------- CKA baseline (non-causal geometry)
@torch.no_grad()
def cka_baseline(model, world, K, site, pos="q", max_pairs=20, rng=None):
    """Linear CKA between rank-matched representations of two domains (succession prompts) at one site and
    position, averaged over domain pairs. site 0 / pos 'opnd' = embedding geometry; pos 'q' at band sites =
    non-causal geometry at the same depth as the transfer measurement."""
    reps = {}
    for d in range(K):
        n = len(world.domains[d])
        T = tens([make_prompt(world, "succ", d, r) for r in range(n - 1)], K)
        _, sites = run(model, T, cache=True)
        reps[d] = sites[site][torch.arange(n - 1, device=DEV), T[pos]].float().cpu().numpy()
    def cka(X, Y):
        X = X - X.mean(0); Y = Y - Y.mean(0)
        return np.linalg.norm(X.T @ Y) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y) + 1e-12)
    pairs = list(combinations(range(K), 2))
    if rng is not None and len(pairs) > max_pairs: pairs = [pairs[i] for i in rng.choice(len(pairs), max_pairs, replace=False)]
    vals = [cka(reps[a][:min(len(reps[a]), len(reps[b]))], reps[b][:min(len(reps[a]), len(reps[b]))]) for a, b in pairs]
    return float(np.mean(vals)) if vals else np.nan

## 1. Ground-truth gate (hard gate)
Planted models are trained on the same data as the grid (K = 2, parametric only), with routing fixed by construction. For each planted share f, the Abstraction Index is computed exactly as it will be for the grid models.

**Pass criteria, fixed before any run:**
1. Every planted model learns every cell to at least 90%. Otherwise the gate is inconclusive.
2. **Separation:** every f = 1 model has a higher AI than every f = 0 model.
3. **Tracking:** Spearman ρ(f, AI) ≥ 0.8 over all planted models.

If the gate fails, the metric needs fixing before anything else runs. The grid cell checks this.

In [ ]:
from scipy.stats import spearmanr

def gate_model(f, seed):
    path = f"{RESULTS_DIR}/gate/f{f:.2f}_s{seed}.json"
    if os.path.exists(path): return json.load(open(path))
    t0 = time.time()
    torch.manual_seed(seed); random.seed(seed)
    w = World(seed, ctx_n=CTX_N)
    cells = [(op, d) for d in range(GATE_K) for op in OPS]
    g = GATE_SIZE
    m = Model(g["L"], g["d"], g["H"]).to(DEV)
    m.route = planted_route(g["L"], g["H"], m.dm, GATE_K, f)
    train(m, Sampler(w, GATE_K, cells, 0.0, seed), GATE_STEPS, g["lr"], BATCH)
    acc = {f"{op}/{d}": accuracy(m, cell_prompts(w, op, d), GATE_K) for op, d in cells}
    circ = measure_A(m, w, GATE_K, cells, np.random.default_rng(seed), EAP_REPS, EAP_PER_REP, EAP_CORR, EAP_IG_STEPS)
    out = dict(f=f, seed=seed, acc=acc, min_acc=min(acc.values()), A=abstraction_index(circ),
               A_topk={k: abstraction_index(circ, "topk", k) for k in (50, 100, 200, 500)}, minutes=(time.time() - t0) / 60)
    save(path, out)
    return json.load(open(path))     # same (JSON) types whether fresh or resumed

GATE = []
for f in GATE_FS:
    for s in GATE_SEEDS:
        r = gate_model(f, s); GATE.append(r)
        nz = lambda v: float("nan") if v is None else v
        print(f"f={f:.2f} seed={s}: min cell acc {r['min_acc']:.2f}  AI {nz(r['A']['AI']):.3f}  "
              f"Δop {nz(r['A']['d_op']):.3f}  Δdom {nz(r['A']['d_dom']):.3f}  G {nz(r['A']['G']):.3f}  "
              f"replicate overlap {nz(r['A']['R_mean']):.2f}  ({r['minutes']:.1f} min)")

ai = lambda r, kind="soft", k=None: (r["A"] if kind == "soft" else r["A_topk"][str(k)])["AI"]
learned = all(r["min_acc"] >= ACC_GATE for r in GATE)
f0 = [ai(r) for r in GATE if r["f"] == 0.0]; f1 = [ai(r) for r in GATE if r["f"] == 1.0]
valid = [r for r in GATE if ai(r) is not None]
sep = bool(f0 and f1 and None not in f0 + f1 and min(f1) > max(f0))
rho_f = spearmanr([r["f"] for r in valid], [ai(r) for r in valid]).statistic if len(valid) > 2 else float("nan")
GATE_PASSED = learned and sep and rho_f >= GATE_MIN_SPEARMAN
print(f"\nPlanted models learned all cells: {learned}")
print(f"Separation (f=1 min {min(f1) if f1 else float('nan'):.3f} > f=0 max {max(f0) if f0 else float('nan'):.3f}): {sep}")
print(f"Tracking: Spearman(f, AI) = {rho_f:.2f} (needs >= {GATE_MIN_SPEARMAN})")
for k in (50, 100, 200, 500):
    vk = [r for r in GATE if ai(r, "topk", k) is not None]
    if len(vk) > 2: print(f"  top-{k} edges: Spearman(f, AI) = {spearmanr([r['f'] for r in vk], [ai(r, 'topk', k) for r in vk]).statistic:.2f}")
print("\nGROUND-TRUTH GATE:", "PASSED" if GATE_PASSED else "FAILED")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for r in GATE:
    ax[0].scatter(r["f"], ai(r), c="#2c3e50", s=25)
    ax[1].scatter(r["f"] - 0.01, r["A"]["d_op"], c="#c0392b", s=20); ax[1].scatter(r["f"] + 0.01, r["A"]["d_dom"], c="#7f8c8d", s=20)
ax[0].set_xlabel("planted shared fraction f"); ax[0].set_ylabel("Abstraction Index"); ax[0].set_title("Gate: AI should rise with f")
ax[1].set_xlabel("planted shared fraction f"); ax[1].set_title("Δop (red) and Δdom (grey)")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/gate.png", dpi=130); plt.show()

## 2. The K × p × size × seed grid
Each model trains with a cosine schedule, using mixed precision on GPU. At ~20 log-spaced checkpoints it records trained-cell, held-out-cell and in-context accuracy, plus a cheap version of measurement B for the H2 trajectories. At the end it runs full A, B and C and the geometry baselines, then writes one JSON file.

**Held-out cells.** These are two operation–domain cells that are never trained. They rotate with the seed and are identical across K ≥ 3, so those models are tested on the same held-out items. Domain 0's succession and predecessor are never held out, because domain 0 is the DAS anchor. With K = 2, the second held-out cell is comparison on domain 0, so each domain keeps at least two trained operations; otherwise measurement B would have nothing to swap.

In [ ]:
def held_out_cells(K, seed):
    o1, o2 = OPS[(seed + 1) % 3], OPS[(seed + 2) % 3]
    if K == 2:   # keep >= 2 operations per domain; domain 0 (the DAS anchor) only ever loses comparison
        return [(o1, 1), ("cmp", 0)] if o1 != "cmp" else [(o1, 1), ("pred", 1)]
    return [(o1, 1), (o2, 2)]

def cell_acc(model, world, K, cells, rng, cap=64):
    out = {}
    for op, d in cells:
        ps = cell_prompts(world, op, d, rng=rng, n_ctx=cap)
        if len(ps) > cap: ps = [ps[i] for i in rng.choice(len(ps), cap, replace=False)]
        out[(op, d)] = accuracy(model, ps, K)
    return out

def run_grid_model(K, p, size, seed):
    rid = f"K{K}_p{p:.2f}_{size}_s{seed}"
    path = f"{RESULTS_DIR}/grid/{rid}.json"
    if os.path.exists(path): return json.load(open(path)), False
    t0 = time.time()
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    w = World(seed, ctx_n=CTX_N)
    held = held_out_cells(K, seed)
    par_cells = [(op, d) for d in range(K) for op in OPS]
    trained = [c for c in par_cells if c not in held]
    ctx_cells = [(op, "ctx") for op in OPS]
    cfg = SIZES[size]; steps = TRAIN_STEPS[size]
    m = Model(cfg["L"], cfg["d"], cfg["H"]).to(DEV)
    traj = []
    rng_t = np.random.default_rng(seed + 7)
    train_set = set(trained) | (set(ctx_cells) if p > 0 else set())

    def on_ckpt(step, model):
        model.requires_grad_(False)
        acc = cell_acc(model, w, K, trained + held + (ctx_cells if p > 0 else []), rng_t)
        pp, pc = measure_B(model, w, K, train_set, rng_t, include_ctx=p > 0, **TRAJ_B)
        traj.append(dict(step=step, acc_trained=float(np.mean([acc[c] for c in trained])),
                         acc_held=float(np.mean([acc[c] for c in held])),
                         acc_ctx=float(np.mean([acc[c] for c in ctx_cells])) if p > 0 else None,
                         peakT_pp=peak(profile(pp, MIN_WITHIN)["T"]) if pp else 0.0,
                         peakT_pc=peak(profile(pc, MIN_WITHIN)["T"]) if pc else None))
        model.requires_grad_(True)

    train(m, Sampler(w, K, trained, p, seed), steps, cfg["lr"], BATCH, log_checkpoints(steps, N_CKPT), on_ckpt)
    t_train = time.time() - t0
    m.requires_grad_(False)
    rng = np.random.default_rng(seed + 11)
    acc = {c: accuracy(m, cell_prompts(w, c[0], c[1], rng=rng, n_ctx=200), K) for c in par_cells + ctx_cells}
    ok = {c for c, a in acc.items() if a >= ACC_GATE and c not in held}
    circ = measure_A(m, w, K, sorted(ok, key=str), rng, EAP_REPS, EAP_PER_REP, EAP_CORR, EAP_IG_STEPS)
    circ_par = {c: v for c, v in circ.items() if c[1] != "ctx"}
    A = dict(soft=abstraction_index(circ_par), topk={k: abstraction_index(circ_par, "topk", k) for k in (50, 100, 200, 500)},
             soft_with_ctx=abstraction_index(circ))
    pp, pc = measure_B(m, w, K, ok, rng, B_N_BASE, B_MAX_TARGETS, B_MAX_SOURCES, include_ctx=True)
    Ppp, Ppc = profile(pp, MIN_WITHIN), profile(pc, MIN_WITHIN)
    B = dict(pp=None if Ppp is None else dict(**{k: v for k, v in Ppp.items()}, peak=peak(Ppp["T"]), band=band(Ppp["T"], T_THRESH), n=len(pp)),
             pc=None if Ppc is None else dict(**{k: v for k, v in Ppc.items()}, peak=peak(Ppc["T"]), band=band(Ppc["T"], T_THRESH), n=len(pc)))
    C = measure_C(m, w, K, ok, rng, DAS_RANK, DAS_STEPS, DAS_LR, seed, DAS_TEST_N)
    band_site = int(np.nanargmax(Ppp["T"])) if Ppp is not None and np.any(~np.isnan(Ppp["T"])) else cfg["L"] // 2
    base = dict(cka_band=cka_baseline(m, w, K, band_site, "q", rng=rng) if K > 1 else None,
                cka_embed=cka_baseline(m, w, K, 0, "opnd", rng=rng) if K > 1 else None, band_site=band_site)
    out = dict(run_id=rid, K=K, p=p, size=size, seed=seed, steps=steps, held=[list(c) for c in held],
               acc={f"{c[0]}/{c[1]}": a for c, a in acc.items()}, ok=[f"{c[0]}/{c[1]}" for c in sorted(ok, key=str)],
               acc_trained=float(np.mean([acc[c] for c in trained])), acc_held=float(np.mean([acc[c] for c in held])),
               acc_ctx=float(np.mean([acc[c] for c in ctx_cells])), A=A, B=B, C=C, rho=rho_summary(C), baselines=base,
               traj=traj, minutes_train=t_train / 60, minutes_total=(time.time() - t0) / 60)
    save(path, out)
    return json.load(open(path)), True

if not GATE_PASSED and not OVERRIDE_GATE:
    print("The ground-truth gate did not pass, so the grid will not run (blueprint: hard gate). "
          "Fix the metric, or set OVERRIDE_GATE = True to run anyway with unvalidated results.")
else:
    if not GATE_PASSED: print("*** OVERRIDE_GATE is set: running the grid although the gate failed. Results are unvalidated. ***")
    configs = [(K, p, size, s) for size in SIZES_RUN for s in SEEDS for K in KS for p in PS]
    done = sum(os.path.exists(f"{RESULTS_DIR}/grid/K{K}_p{p:.2f}_{z}_s{s}.json") for K, p, z, s in configs)
    print(f"{len(configs)} models in this grid; {done} already finished.")
    for K, p, size, s in configs:
        if (time.time() - T_SESSION) / 60 > TIME_BUDGET_MIN:
            print(f"Time budget of {TIME_BUDGET_MIN} min reached. Rerun the notebook to resume."); break
        r, new = run_grid_model(K, p, size, s)
        if new:
            a = r["A"]["soft"]["AI"]; pk = r["B"]["pp"]["peak"] if r["B"]["pp"] else None
            print(f"{r['run_id']:22s} trained {r['acc_trained']:.2f} held-out {r['acc_held']:.2f} ctx {r['acc_ctx']:.2f} | "
                  f"AI {a if a is not None else float('nan'):.2f}  peak T {pk if pk is not None else float('nan'):.2f} | {r['minutes_total']:.1f} min")

## 3. Analyses
Every finished model on Drive is loaded, so these cells can be rerun at any time, including on a partial grid. With the 16-model pilot, treat every number as a pilot estimate. The confirmatory versions need the full 120-model grid.

In [ ]:
import pandas as pd
rows = []
for fn in sorted(os.listdir(f"{RESULTS_DIR}/grid")):
    if not fn.endswith(".json"): continue
    r = json.load(open(f"{RESULTS_DIR}/grid/{fn}"))
    rho_par = [v["rho"] for k, v in r["rho"].items() if v.get("informative") and v.get("rho") is not None and not k.endswith("/ctx") and k != "pred/0"]
    rho_ctx = [v["rho"] for k, v in r["rho"].items() if v.get("informative") and v.get("rho") is not None and k.endswith("/ctx")]
    rows.append(dict(run_id=r["run_id"], K=r["K"], p=r["p"], size=r["size"], seed=r["seed"],
                     acc_trained=r["acc_trained"], acc_held=r["acc_held"], acc_ctx=r["acc_ctx"],
                     AI=r["A"]["soft"]["AI"], d_op=r["A"]["soft"]["d_op"], d_dom=r["A"]["soft"]["d_dom"], G=r["A"]["soft"]["G"],
                     R=r["A"]["soft"]["R_mean"], peakT=r["B"]["pp"]["peak"] if r["B"]["pp"] else np.nan,
                     peakT_ctx=r["B"]["pc"]["peak"] if r["B"]["pc"] else np.nan,
                     rho_par=np.mean(rho_par) if rho_par else np.nan, rho_ctx=np.mean(rho_ctx) if rho_ctx else np.nan,
                     cka_band=r["baselines"]["cka_band"], cka_embed=r["baselines"]["cka_embed"], traj=r["traj"], steps=r["steps"]))
df = pd.DataFrame(rows)
print(f"{len(df)} finished models")
n_clip = sum(json.load(open(f"{RESULTS_DIR}/grid/{fn}"))["A"]["soft"].get("clipped", False)
             for fn in os.listdir(f"{RESULTS_DIR}/grid") if fn.endswith(".json"))
if n_clip: print(f"AI was clipped to [0, 1] in {n_clip} model(s), where Δdom or Δop was negative; see AI_raw in the JSON.")
if len(df):
    show = ["K", "p", "size", "seed", "acc_trained", "acc_held", "acc_ctx", "AI", "d_op", "d_dom", "R", "peakT", "peakT_ctx", "rho_par", "rho_ctx"]
    print(df[show].sort_values(["size", "K", "p", "seed"]).round(2).to_string(index=False))

def boot_ci(fn, n, B=2000, seed=0):
    r = np.random.default_rng(seed); vals = []
    for _ in range(B):
        v = fn(r.integers(0, n, n))
        if v is not None and np.isfinite(v): vals.append(v)
    return (np.percentile(vals, 2.5), np.percentile(vals, 97.5)) if len(vals) > 20 else (np.nan, np.nan)

### H1: does sharing grow with the number of training domains K?
The confirmatory test is a mixed-effects regression of the Abstraction Index on log₂K, p and size, with seed as a random effect. It is falsified if the K-trend's rank-correlation CI includes 0. Peak cross-domain transfer is tested the same way.

**Δop is tested as well.** AI is a ratio, so it only ranks models. In testing, an ordinary trained model had Δdom ≈ 0, because domain identity lived in token embeddings rather than in separate components, and that pushed AI to about 0.95. When Δdom is that small, AI saturates and can't show a trend. Δop, the operation-specific overlap above the generic baseline, carries the magnitude.

In [ ]:
H1 = {}
if len(df) >= 4:
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for metric, a in (("AI", ax[0]), ("d_op", ax[2]), ("peakT", ax[1])):
        d = df.dropna(subset=[metric])
        if len(d) < 4: continue
        rs = spearmanr(d["K"], d[metric]).statistic
        lo, hi = boot_ci(lambda i: spearmanr(d["K"].values[i], d[metric].values[i]).statistic, len(d))
        H1[metric] = dict(spearman=rs, ci=[lo, hi], n=len(d))
        print(f"{metric}: Spearman with K = {rs:.2f}, 95% CI [{lo:.2f}, {hi:.2f}] over {len(d)} models -> "
              f"{'falsifies H1 (CI includes 0)' if lo <= 0 <= hi else 'consistent with H1' if lo > 0 else 'opposite direction'}")
        try:
            import statsmodels.formula.api as smf
            d2 = d.assign(log2K=np.log2(d["K"]))
            form = f"{metric} ~ log2K + p" + (" + C(size)" if d2["size"].nunique() > 1 else "")
            fit = smf.mixedlm(form, d2, groups=d2["seed"]).fit()
            H1[metric]["mixed_log2K"] = [fit.params["log2K"], *fit.conf_int().loc["log2K"].tolist()]
            print(f"   mixed model: slope on log2 K = {fit.params['log2K']:.3f}, 95% CI [{fit.conf_int().loc['log2K'][0]:.3f}, {fit.conf_int().loc['log2K'][1]:.3f}]")
        except Exception as e:
            print(f"   mixed model not fitted ({type(e).__name__}); needs more models or variation")
        for (pp_, sz), g in d.groupby(["p", "size"]):
            mk = g.groupby("K")[metric].agg(["mean", "std"])
            a.errorbar(mk.index, mk["mean"], yerr=mk["std"], marker="o", capsize=3, label=f"p={pp_}, {sz}")
        a.set_xscale("log", base=2); a.set_xlabel("training domains K"); a.set_ylabel(metric); a.legend(fontsize=8)
    ax[0].set_title("Abstraction Index vs K"); ax[1].set_title("Peak cross-domain transfer T vs K"); ax[2].set_title("Δop (operation-specific overlap) vs K")
    if len(df) and df["d_dom"].abs().median() < 0.05:
        print("\nNote: Δdom is near 0 in these models (domain identity lives in token embeddings, not in separate components), "
              "so AI saturates near 1 whenever Δop > 0. Read the K-trend from Δop and peak T, not from AI.")
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/h1.png", dpi=130); plt.show()
else:
    print("Need at least 4 finished models.")

### H2: does the band form abruptly, at the point where held-out cells start to generalise?
Band strength is the peak cross-domain transfer T at each checkpoint, from the cheap B. The onset of each trajectory is the best single change point, from a two-segment piecewise-constant least-squares fit over checkpoints. A trajectory with less than a 0.2 rise has **no onset**. H2 is falsified if there is no change point in band strength, or if its onset differs from the held-out-accuracy onset by more than 20% of training.

In [ ]:
def change_point(steps, y, min_rise=0.2):
    y = np.asarray(y, float); steps = np.asarray(steps)
    if len(y) < 4 or np.nanmax(y) - np.nanmin(y) < min_rise: return None
    best, k_best = np.inf, None
    for k in range(1, len(y)):
        sse = ((y[:k] - y[:k].mean()) ** 2).sum() + ((y[k:] - y[k:].mean()) ** 2).sum()
        if sse < best: best, k_best = sse, k
    return int(steps[k_best])

H2_rows = []
for _, r in df.iterrows():
    st = [t["step"] for t in r["traj"]]
    on_band = change_point(st, [t["peakT_pp"] for t in r["traj"]])
    on_held = change_point(st, [t["acc_held"] for t in r["traj"]])
    on_tr = change_point(st, [t["acc_trained"] for t in r["traj"]])
    H2_rows.append(dict(run_id=r["run_id"], K=r["K"], p=r["p"], onset_band=on_band, onset_held=on_held, onset_trained=on_tr,
                        diff_frac=(on_band - on_held) / r["steps"] if on_band and on_held else None))
H2 = pd.DataFrame(H2_rows)
if len(H2):
    print(H2.to_string(index=False))
    both = H2.dropna(subset=["diff_frac"])
    print(f"\nModels with a band onset: {H2['onset_band'].notna().sum()}/{len(H2)}; with a held-out onset: {H2['onset_held'].notna().sum()}/{len(H2)}")
    if len(both) >= 3:
        lo, hi = boot_ci(lambda i: both["diff_frac"].values[i].mean(), len(both))
        print(f"Mean onset difference (band - held-out) = {both['diff_frac'].mean():.2f} of training, 95% CI [{lo:.2f}, {hi:.2f}]; "
              f"H2 falsified if beyond ±0.20")
    else:
        print("Too few models with both onsets to test H2. If held-out cells never generalise, H2 cannot be tested in this grid.")
    sel = df.sort_values(["K", "p"]).groupby(["K", "p"]).head(1)
    fig, axs = plt.subplots(1, len(sel), figsize=(3.2 * len(sel), 3), sharey=True, squeeze=False)
    for a, (_, r) in zip(axs[0], sel.iterrows()):
        st = [t["step"] for t in r["traj"]]
        a.plot(st, [t["peakT_pp"] for t in r["traj"]], label="band strength (peak T)", c="#2c3e50")
        a.plot(st, [t["acc_held"] for t in r["traj"]], label="held-out acc", c="#c0392b")
        a.plot(st, [t["acc_trained"] for t in r["traj"]], label="trained acc", c="#95a5a6", ls=":")
        a.set_xscale("log"); a.set_title(f"K={r['K']} p={r['p']} s{r['seed']}", fontsize=9); a.set_xlabel("step")
    axs[0][0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/h2.png", dpi=130); plt.show()

### H4 in miniature: does in-context training pull ordering into the shared band?
This compares parametric → in-context transfer (B), and subspace transfer from the anchor into in-context lists (C, informative targets only), across p. With p = 0 the model never sees lists, so in-context cells fail the accuracy gate and drop out.

In [ ]:
if len(df):
    g = df.groupby(["p", "K"])[["acc_ctx", "peakT_ctx", "rho_ctx", "peakT", "rho_par"]].mean().round(2)
    print(g.to_string())
    print("\nH4 in miniature is supported if peakT_ctx and rho_ctx at p > 0 approach the parametric values (>= 0.5 at band layers).")

### H5, Part 1 version: does transfer on known operations predict generalisation to held-out cells?
The predictor is model-level peak cross-domain transfer T. The outcome is held-out-cell accuracy. Partial Spearman controls for the design factors (log₂K, p, size), behavioural ability (trained-cell accuracy) and two geometry baselines: CKA at the band site, and embedding-level CKA. H5 is falsified if the partial correlation's CI includes 0. The null comes from shuffling the outcome across models.

In [ ]:
def partial_spearman(x, y, Z):
    from scipy.stats import rankdata
    rx, ry = rankdata(x), rankdata(y)
    Zr = np.column_stack([np.ones(len(x))] + [rankdata(z) for z in Z.T]) if Z.size else np.ones((len(x), 1))
    res = lambda v: v - Zr @ np.linalg.lstsq(Zr, v, rcond=None)[0]
    a, b = res(rx), res(ry)
    return float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else np.nan

H5 = {}
d = df.dropna(subset=["peakT", "acc_held"]).copy()
if len(d) >= 6:
    d["log2K"] = np.log2(d["K"]); d["size_n"] = (d["size"] == "large").astype(float)
    covs = [c for c in ["log2K", "p", "size_n", "acc_trained", "cka_band", "cka_embed"] if d[c].notna().all() and d[c].nunique() > 1]
    x, y, Z = d["peakT"].values, d["acc_held"].values, d[covs].values.astype(float)
    rs = spearmanr(x, y).statistic
    pr = partial_spearman(x, y, Z)
    lo, hi = boot_ci(lambda i: partial_spearman(x[i], y[i], Z[i]), len(d))
    perm = np.random.default_rng(0)
    null = [partial_spearman(x, perm.permutation(y), Z) for _ in range(2000)]
    pval = float(np.mean(np.abs(null) >= abs(pr))) if np.isfinite(pr) else np.nan
    H5 = dict(spearman=rs, partial=pr, ci=[lo, hi], perm_p=pval, covariates=covs, n=len(d))
    print(f"Spearman(peak T, held-out acc) = {rs:.2f} over {len(d)} models")
    print(f"Partial Spearman controlling for {covs} = {pr:.2f}, 95% CI [{lo:.2f}, {hi:.2f}], permutation p = {pval:.3f}")
    if d["acc_held"].max() - d["acc_held"].min() < 0.1:
        print("Held-out accuracy barely varies across models, so H5 has nothing to predict in this grid.")
    plt.figure(figsize=(4.8, 3.8))
    for K_, g_ in d.groupby("K"): plt.scatter(g_["peakT"], g_["acc_held"], label=f"K={K_}", s=25)
    plt.xlabel("peak cross-domain transfer T"); plt.ylabel("held-out-cell accuracy"); plt.legend(fontsize=8)
    plt.title("H5 (Part 1): transfer vs generalisation"); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/h5.png", dpi=130); plt.show()
else:
    print("Need at least 6 finished models with a transfer estimate.")

## Summary

In [ ]:
summary = dict(gate=dict(passed=GATE_PASSED, separation=sep, spearman_f_AI=rho_f, learned=learned),
               n_models=len(df), H1=H1, H5=H5,
               H2=dict(n_band_onset=int(H2["onset_band"].notna().sum()) if len(H2) else 0,
                       n_held_onset=int(H2["onset_held"].notna().sum()) if len(H2) else 0))
save(f"{RESULTS_DIR}/summary.json", summary)
print(json.dumps(to_json(summary), indent=1))
print(f"\nEverything is in {RESULTS_DIR}: per-model JSONs (grid/), gate models (gate/), plots and summary.json.")